<a href="https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### My lane as an ML task

I chose **Lane 2: Refresh / Content Opportunity Scoring**. I frame it primarily as a **ranking/scoring problem** rather than simply a classification problem.

The decision is: **which pages should a content team review first?** Each page can receive a priority score and the pages can then be ranked from highest to lowest priority. The output supports a human action such as refresh, expand, protect, prune, or monitor.

Classification may still be used inside the system to estimate a useful outcome, such as the likelihood that a page is declining, but the final product is a ranked queue. This makes ranking/scoring the better description of the overall ML task.

### Target or proxy

For the starter dataset, my provisional proxy is whether a page is observed as **declining**:

`is_declining = 1 if trend_direction == "down", otherwise 0`

This label comes from an **observed trend category in the dataset**, rather than from my own hand-written priority rule. A model could estimate a score related to this observed outcome and use that score to help rank pages for review.

However, being declining is **not the same as proving that refreshing the page will produce a positive result**. A declining page may have low business importance, while another page may deserve review for a different reason. Therefore, this is a provisional proxy for the starter-data exercise, not a complete definition of content opportunity.

For later work with time-window data, I would prefer an outcome measured in a future period so that the model uses earlier information to predict a later observed outcome.

In [2]:
!git clone https://github.com/mbinaqeel-analyst/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 129 (delta 43), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.85 MiB | 18.75 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank-ml-internship


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


### Success metric

My primary success metric is **Precision@50**.

The practical scenario is that a content team has limited time and can review only a small number of pages first. Precision@50 answers: **of the top 50 pages recommended by my ranking, how many match the defined target or proxy?**

A higher Precision@50 means the top of the queue is more useful for prioritization. I would compare this result against a transparent hand-written baseline rather than assuming that an ML model is useful simply because it is more complex.

For this starter-data exercise, I will treat improvement over the baseline as evidence that the learned ranking may provide better decision support. The metric does not prove that refreshing those pages will improve Google rankings or traffic.

In [7]:
total_pages = len(df)
review_capacity = 50

print(f"Total pages: {total_pages:,}")
print(f"Pages reviewed at K=50: {review_capacity}")
print(f"Share of inventory reviewed: {review_capacity / total_pages:.2%}")

Total pages: 30,000
Pages reviewed at K=50: 50
Share of inventory reviewed: 0.17%


### The unit of analysis

The unit of analysis is **one page**.

Each row in the starter dataset represents one anonymized content page and contains measurements and attributes associated with that page. The ranking system will assign each row/page a priority score.

The output will therefore also be page-level: one scored row for each page, which can then be sorted into a ranked review queue.

In [6]:
columns_to_show = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction",
]

page_level_data = df[columns_to_show].head(10)

print("Unit of analysis: one row = one anonymized content page")
page_level_data

Unit of analysis: one row = one anonymized content page


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction
0,187,20,3803,10.6,0.76,3221.0,down
1,445,25,15320,20.3,0.05,2481.0,down
2,141,20,12581,36.5,0.09,3515.0,down
3,463,22,11751,6.2,0.49,NaN,stable
4,263,14,19140,44.0,0.13,2803.0,down
5,147,20,3970,8.5,0.03,3080.0,down
6,90,20,20,7.0,0.00,3059.0,down
7,445,22,1724,21.2,0.06,NaN,stable
8,90,20,32574,46.0,0.09,3807.0,down
9,257,104,1240,4.9,0.16,NaN,down


### Why ML may beat a fixed rule here

A fixed rule can provide a useful baseline, for example: review a page if it is old, has low CTR, and is declining. However, page priority is likely to depend on several signals interacting together.

A page's age, freshness, visibility, average position, CTR, content length, engagement, and other available signals may not have one simple set of thresholds that works well for every page. For example, a low CTR can mean something different for a highly visible page than for a page with very few impressions.

ML may help combine multiple signals and learn more complex patterns than a single if-statement. However, I will not assume ML is automatically better. It earns its use only if it produces a meaningfully better ranking on an honest validation set than a transparent fixed-rule baseline.

The final output remains decision support for a human reviewer, not an automatic guarantee that a recommended page should be refreshed.

In [4]:
candidate_signals = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

print("Candidate signals that could contribute to a page-level priority score:")
print(candidate_signals)

print("\nNumber of candidate signals:", len(candidate_signals))

Candidate signals that could contribute to a page-level priority score:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate']

Number of candidate signals: 7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.